### Data Ingestion

In [1]:
### Document Structure
from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Shreya Lakhmani",
        "date_created":"2026-08-30"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Shreya Lakhmani', 'date_created': '2026-08-30'}, page_content='this is the main text content I am using to create RAG')

In [9]:
### Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [10]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)
        
print("✅ Sample text files created!")

✅ Sample text files created!


In [12]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [14]:
##Directory Loader
from langchain_community.document_loaders import DirectoryLoader

### load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files
    loader_cls=TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\

In [16]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader

##load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ##Pattern to match files
    loader_cls=PyMuPDFLoader, ##loader class to use
    show_progress=False
)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-07T15:21:59+00:00', 'source': '..\\data\\pdf\\Achyuta & Sheetal wedding card (1).pdf', 'file_path': '..\\data\\pdf\\Achyuta & Sheetal wedding card (1).pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'Achyuta & Sheetal wedding card', 'author': 'Aishwarya', 'subject': '', 'keywords': 'DAG85yLlcCQ,BAG5x998TDo,0', 'moddate': '2026-01-07T15:21:52+00:00', 'trapped': '', 'modDate': "D:20260107152152+00'00'", 'creationDate': "D:20260107152159+00'00'", 'page': 0}, page_content='A\nc\nh\ny\nu\nt\na\n \n&\n \nS\nh\ne\ne\nt\na\nl\n#\nS\nH\nE\nE\nf\no\nu\nn\nd\nh\ne\nr\nK\nR\nI\nS\nH\nN\nA\nहरि ॐ\nAchyuta & Sheetal\n11th March 2026\nHotel Krinoscco, Ayodhya\nवक्रतुण्ड महाकाय सूर्यकोटि समप्रभ:\nनिर्विघ्नं कुरु मे देव, सर्वकार्येषु सर्वदा ।।\n ।'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-07T15:21:59+00:00', 'source': '..\\data\\pdf\\Achyuta & Sheetal wedding card (

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [23]:
chunks=split_documents(pdf_documents)
chunks


Split 292 documents into 288 chunks

Example chunk:
Content: A
c
h
y
u
t
a
 
&
 
S
h
e
e
t
a
l
#
S
H
E
E
f
o
u
n
d
h
e
r
K
R
I
S
H
N
A
हरि ॐ
Achyuta & Sheetal
11th March 2026
Hotel Krinoscco, Ayodhya
वक्रतुण्ड महाकाय सूर्यकोटि समप्रभ:
निर्विघ्नं कुरु मे देव, सर...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-07T15:21:59+00:00', 'source': '..\\data\\pdf\\Achyuta & Sheetal wedding card (1).pdf', 'file_path': '..\\data\\pdf\\Achyuta & Sheetal wedding card (1).pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'Achyuta & Sheetal wedding card', 'author': 'Aishwarya', 'subject': '', 'keywords': 'DAG85yLlcCQ,BAG5x998TDo,0', 'moddate': '2026-01-07T15:21:52+00:00', 'trapped': '', 'modDate': "D:20260107152152+00'00'", 'creationDate': "D:20260107152159+00'00'", 'page': 0}


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-07T15:21:59+00:00', 'source': '..\\data\\pdf\\Achyuta & Sheetal wedding card (1).pdf', 'file_path': '..\\data\\pdf\\Achyuta & Sheetal wedding card (1).pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'Achyuta & Sheetal wedding card', 'author': 'Aishwarya', 'subject': '', 'keywords': 'DAG85yLlcCQ,BAG5x998TDo,0', 'moddate': '2026-01-07T15:21:52+00:00', 'trapped': '', 'modDate': "D:20260107152152+00'00'", 'creationDate': "D:20260107152159+00'00'", 'page': 0}, page_content='A\nc\nh\ny\nu\nt\na\n \n&\n \nS\nh\ne\ne\nt\na\nl\n#\nS\nH\nE\nE\nf\no\nu\nn\nd\nh\ne\nr\nK\nR\nI\nS\nH\nN\nA\nहरि ॐ\nAchyuta & Sheetal\n11th March 2026\nHotel Krinoscco, Ayodhya\nवक्रतुण्ड महाकाय सूर्यकोटि समप्रभ:\nनिर्विघ्नं कुरु मे देव, सर्वकार्येषु सर्वदा ।।\n ।'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-07T15:21:59+00:00', 'source': '..\\data\\pdf\\Achyuta & Sheetal wedding card (